In [5]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import os
import random
import time
import copy
import warnings
import numpy as np
from PIL import Image, ImageOps
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models
from tqdm import tqdm

# Suppress harmless PIL/Tiff metadata user warnings
warnings.filterwarnings("ignore", category=UserWarning, module="PIL")

# Import DirectML for Intel / AMD Integrated GPU & NPU Acceleration
try:
    import torch_directml
    HAS_DIRECTML = True
except ImportError:
    HAS_DIRECTML = False

# ==========================================
# 1. REPRODUCIBILITY & SEEDING
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==========================================
# 2. CONFIGURATION & HYPERPARAMETERS
# ==========================================
DATA_DIR = "./Data"  # Directory containing your 16 class folders
SAVE_PATH = "rvl_cdip_efficientnet_best.pth"
ONNX_PATH = "rvl_cdip_efficientnet_best.onnx"

# SPEED & ACCURACY BALANCING CONTROLS
USE_FAST_MODE = False     # Set True for subsampled training
MAX_SAMPLES = 20000       # Subsampled to 20,000 images for higher dataset coverage

BATCH_SIZE = 64           # Increased batch size to process 20k images faster
NUM_EPOCHS = 5           # 10 epochs
VAL_SPLIT = 0.2           # 80% Train (16,000), 20% Validation (4,000)

# System Hardware Detection (CUDA -> DirectML -> CPU)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("--> Hardware Acceleration: NVIDIA CUDA GPU Detected")
elif HAS_DIRECTML and torch_directml.is_available():
    DEVICE = torch_directml.device()
    print("--> Hardware Acceleration: DirectML (Integrated GPU / NPU) Detected")
else:
    DEVICE = torch.device("cpu")
    print("--> Hardware Acceleration: CPU Mode")

NUM_WORKERS = 0 if os.name == 'nt' else os.cpu_count()  # 0 on Windows prevents multiproc locks
PIN_MEMORY = False  # DirectML uses shared host memory

print(f"--> Target Device: {DEVICE}")

# ==========================================
# 3. SAFE PIL IMAGE LOADER & VALIDATOR
# ==========================================
def is_valid_image(path):
    """Validates if an image file can be opened without corruption."""
    try:
        with open(path, "rb") as f:
            img = Image.open(f)
            img.verify()
        return True
    except Exception:
        return False

def safe_pil_loader(path):
    """Safely opens images, fixes EXIF orientation, and converts to RGB."""
    try:
        with open(path, "rb") as f:
            img = Image.open(f)
            img = ImageOps.exif_transpose(img)
            return img.convert("RGB")
    except Exception as e:
        print(f"[Warning] Skipping corrupt image {path}: {e}")
        return Image.new("RGB", (260, 260), (0, 0, 0))

# ==========================================
# 4. DATA TRANSFORMS FOR DOCUMENTS
# ==========================================
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((260, 260)),                              # EfficientNet-B2 native size
        transforms.RandomRotation(degrees=3),                       # Slight scan rotation
        transforms.RandomPerspective(distortion_scale=0.1, p=0.3),  # Camera tilt simulation
        transforms.ColorJitter(brightness=0.2, contrast=0.2),     # Scan contrast variance
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "val": transforms.Compose([
        transforms.Resize((260, 260)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

class TransformedSubset(torch.utils.data.Dataset):
    """Applies specific transforms and handles corrupt image indexing gracefully."""
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        try:
            x, y = self.subset[index]
            if self.transform:
                x = self.transform(x)
            return x, y
        except Exception:
            return self.__getitem__(0)

    def __len__(self):
        return len(self.subset)

# ==========================================
# 5. DATASETS & DATALOADERS
# ==========================================
print("--> Scanning and validating dataset files...")
full_dataset = datasets.ImageFolder(
    root=DATA_DIR, 
    loader=safe_pil_loader,
    is_valid_file=is_valid_image  # Pre-filters corrupted TIFF/JPEG files
)
class_names = full_dataset.classes
num_classes = len(class_names)

print(f"--> Discovered {len(full_dataset)} total valid images across {num_classes} classes.")

if USE_FAST_MODE and len(full_dataset) > MAX_SAMPLES:
    indices = list(range(len(full_dataset)))
    random.seed(SEED)
    random.shuffle(indices)
    selected_indices = indices[:MAX_SAMPLES]
    active_dataset = Subset(full_dataset, selected_indices)
    print(f"--> [FAST MODE] Subsampled dataset to {len(active_dataset)} images.")
else:
    active_dataset = full_dataset

val_size = int(len(active_dataset) * VAL_SPLIT)
train_size = len(active_dataset) - val_size

base_train, base_val = random_split(
    active_dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(SEED)
)

train_dataset = TransformedSubset(base_train, transform=data_transforms["train"])
val_dataset = TransformedSubset(base_val, transform=data_transforms["val"])

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

dataloaders = {"train": train_loader, "val": val_loader}

# ==========================================
# 6. MODEL SETUP (EFFICIENTNET-B2)
# ==========================================
model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)

# Replace classification head for 16 classes
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, num_classes)
)

model = model.to(DEVICE)

# Differential learning rates
backbone_params = [p for n, p in model.named_parameters() if "classifier" not in n]
head_params = [p for n, p in model.named_parameters() if "classifier" in n]

# foreach=False is critical to prevent CPU fallback on DirectML
optimizer = optim.AdamW([
    {"params": backbone_params, "lr": 2e-4}, 
    {"params": head_params, "lr": 1e-3}
], weight_decay=1e-2, foreach=False)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# ==========================================
# 7. TRAINING & VALIDATION LOOP
# ==========================================
def train_model(model, criterion, optimizer, scheduler, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        print("-" * 35)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            pbar = tqdm(dataloaders[phase], desc=f"{phase.capitalize():<5}", leave=True)

            for inputs, labels in pbar:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

                batch_acc = torch.sum(preds == labels.data).item() / inputs.size(0)
                pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{batch_acc:.2f}")

            if phase == "train":
                scheduler.step()

            dataset_size = train_size if phase == "train" else val_size
            epoch_loss = running_loss / dataset_size
            epoch_acc = (running_corrects.double() / dataset_size).item()

            print(f"Summary -> {phase.capitalize():<5} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            # Save best performing model weights
            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), SAVE_PATH)
                print(f"--> [NEW BEST] Saved model to '{SAVE_PATH}' (Val Acc: {best_acc:.4f})")

    time_elapsed = time.time() - since
    print(f"\nTraining completed in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Peak Validation Accuracy: {best_acc:.4f}")

    # Load best weights
    model.load_state_dict(best_model_wts)
    
    # Export to ONNX (transfer to CPU first for safe export)
    try:
        model_cpu = model.to(torch.device("cpu"))
        model_cpu.eval()
        dummy_input = torch.randn(1, 3, 260, 260)
        torch.onnx.export(
            model_cpu,
            dummy_input,
            ONNX_PATH,
            input_names=["input"],
            output_names=["output"],
            dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
        )
        print(f"--> Model successfully exported to ONNX format at '{ONNX_PATH}'.")
    except Exception as e:
        print(f"[Warning] Failed to export ONNX model: {e}")

    return model

# ==========================================
# 8. EXECUTION
# ==========================================
if __name__ == "__main__":
    trained_model = train_model(
        model, 
        criterion, 
        optimizer, 
        scheduler, 
        num_epochs=NUM_EPOCHS
    )

--> Hardware Acceleration: DirectML (Integrated GPU / NPU) Detected
--> Target Device: privateuseone:0
--> Scanning and validating dataset files...
--> Discovered 39996 total valid images across 16 classes.

Epoch 1/5
-----------------------------------


Train: 100%|██████████| 500/500 [2:19:58<00:00, 16.80s/it, acc=0.67, loss=1.3855]     


Summary -> Train Loss: 1.4445 Acc: 0.6698


Val  : 100%|██████████| 125/125 [03:42<00:00,  1.78s/it, acc=0.65, loss=1.3723]


Summary -> Val   Loss: 1.1500 Acc: 0.7812
--> [NEW BEST] Saved model to 'rvl_cdip_efficientnet_best.pth' (Val Acc: 0.7812)

Epoch 2/5
-----------------------------------


...

Summary -> Val   Loss: 0.9736 Acc: 0.8521
--> [NEW BEST] Saved model to 'rvl_cdip_efficientnet_best.pth' (Val Acc: 0.8521)

Epoch 5/5
-----------------------------------


Train: 100%|██████████| 500/500 [1:01:19<00:00,  7.36s/it, acc=0.89, loss=0.8417]


Summary -> Train Loss: 0.8190 Acc: 0.9122


Val  : 100%|██████████| 125/125 [03:41<00:00,  1.77s/it, acc=0.84, loss=1.0709]


Summary -> Val   Loss: 0.9657 Acc: 0.8551
--> [NEW BEST] Saved model to 'rvl_cdip_efficientnet_best.pth' (Val Acc: 0.8551)

Training completed in 419m 29s
Peak Validation Accuracy: 0.8551
--> Model successfully exported to ONNX format at 'rvl_cdip_efficientnet_best.onnx'.
